ReAct Agent

think--> Act--> Observe loop, no agent libraries. Just groq+python


Setup

In [ ]:
!pip install groq -q

Importing Dependencies

In [ ]:
import os, re, getpass
from groq import Groq

Groq client


In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Provide your Groq API Key: ")


In [ ]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.3-70b-versatile"

TOOLS

Plain Python function, registered in a dictionary

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a math expression and return the result as a string.

    Args:
        expression (str): Math expression, e.g. "23 * 47" or "(100 + 5) / 3".

    Returns:
        str: Numeric result, or an error message starting with "Error:".
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


def get_weather(city: str) -> str:
    """Return current weather for a city (mocked).

    Args:
        city (str): City name, case-insensitive.

    Returns:
        str: Weather description, or a "no data" message.
    """
    data = {
        "chennai": "32°C, humid, partly cloudy",
        "bangalore": "24°C, pleasant, light rain",
        "delhi": "28°C, hazy",
        "mumbai": "30°C, humid",
    }
    return data.get(city.lower(), f"No weather data for {city}")


def word_count(text: str) -> str:
    """Count whitespace-separated words in a string.

    Args:
        text (str): Input text.

    Returns:
        str: Word count as a string.
    """
    return str(len(text.split()))

In [ ]:
TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather,
    "word_count": word_count
}

In [ ]:
TOOL_DESCRIPTIONS = """
- calculator(expression: str) -> str
    Evaluate a math expression. Example: calculator("23 * 47")
- get_weather(city: str) -> str
    Return current weather for a city. Example: get_weather("Chennai")
- word_count(text: str) -> str
    Count words in text. Example: word_count("hello world")
"""

SYSTEM PROMPT

Defines the Thought / Action / Observation format the parser depends on.

In [ ]:
SYSTEM_PROMPT = f"""You are a ReAct agent that solves problems step by step.

Tools available:
{TOOL_DESCRIPTIONS}

Format (follow exactly):

Thought: <reasoning>
Action: <tool_name>
Action Input: <input string>

After Action, STOP. The system replies with:

Observation: <result>

Continue with another Thought/Action, or finish:

Thought: I now know the final answer.
Final Answer: <answer>

Rules:
- One Thought + Action per turn, then wait.
- Action must be one of: {list(TOOLS.keys())}
- Action Input is a plain string (no quotes).
- Never invent Observations.
"""

In [ ]:
print(SYSTEM_PROMPT)

Parser

In [ ]:
def parse_response(text: str):
    """Parse LLM output into ('final', answer) | ('action', name, input) | ('error', msg)."""
    if m := re.search(r"Final Answer:\s*(.+)", text, re.DOTALL):
        return ("final", m.group(1).strip())

    a = re.search(r"Action:\s*(.+)", text)
    i = re.search(r"Action Input:\s*(.+)", text)
    if a and i:
        return ("action", a.group(1).strip(), i.group(1).strip().strip('"').strip("'"))

    return ("error", "Could not parse Action or Final Answer.")

Agent loop



**🤖 ReAct Agent Flow**

```
   User Question
        │
        ▼
   ┌─────────┐
   │  THINK  │ ◄──────────┐
   │  (LLM)  │            │
   └────┬────┘            │
        │                 │
        ▼                 │
   ┌─────────┐            │
   │  PARSE  │            │
   └────┬────┘            │
        │                 │
   ┌────┴────┐            │
   │         │            │
   ▼         ▼            │
 Final?    Tool Call      │
   │         │            │
   │         ▼            │
   │     ┌──────┐         │
   │     │ ACT  │         │
   │     └──┬───┘         │
   │        │             │
   │        ▼             │
   │   Observation ───────┘
   │
   ▼
 ✅ Answer
```

**Loop:** Think → Act → Observe → repeat until final answer (or max steps).

In [ ]:
def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    """Run the ReAct loop until Final Answer or max_steps."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    if verbose:
        print(f"🧑 {question}\n" + "=" * 60)

    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0,
            stop=["Observation:"],
        )
        out = resp.choices[0].message.content.strip()
        messages.append({"role": "assistant", "content": out})
        if verbose:
            print(f"\n--- Step {step} ---\n🤖 {out}")

        parsed = parse_response(out)

        if parsed[0] == "final":
            if verbose: print(f"\n✅ {parsed[1]}")
            return parsed[1]
        if parsed[0] == "error":
            if verbose: print(f"⚠️ {parsed[1]}")
            return parsed[1]

        _, name, arg = parsed
        if name in TOOLS:
            try:    obs = TOOLS[name](arg)
            except Exception as e: obs = f"Error running {name}: {e}"
        else:
            obs = f"Error: unknown tool '{name}'. Available: {list(TOOLS.keys())}"

        if verbose: print(f"🔧 {obs}")
        messages.append({"role": "user", "content": f"Observation: {obs}"})

    return "Agent stopped: max_steps reached."

TRY IT

In [ ]:
run_agent("What is 234*17, then - 3?")

In [ ]:
run_agent("What's the weather in Chennai?")